In [1]:
import PhidgetUtils
import numpy as np
import tkinter as tk
from tkinter import ttk
import uuid
from math import ceil
import os

from time import sleep, time

In [2]:
acc = PhidgetUtils.PhidgetAccelerometer(data_rate=60)

while not acc.is_attached():
    sleep(0.1)

print("Accelerometer is attached!")

Accelerometer is attached!


In [3]:
def magnitude(v):
    return v[0] * v[0] + v[1] * v[1] + v[2] * v[2]

def sign(v):
    return 1 if v > 0 else -1

In [4]:
RECORDING = False
frequency = 60

folder = f"./datastreams/{time()}"
os.makedirs(folder, exist_ok=True)

dtype = [('timestamp', 'int32'), ('acceleration', 'float32')]

data_size_from_mins = lambda mins : int(3600 * mins)

data_size = data_size_from_mins(0.5)
data = np.memmap(f"{folder}/0.dat", dtype=dtype, mode='w+', shape=(data_size,))
data_index = 0
file_count = 1


prev_acc = -1
prev_pressure = -1

root = tk.Tk()
root.title("Accelerometer + Barometer Measurements")
root.geometry("400x250")
root.resizable(False, False)

''

In [5]:
start_button = stop_button = None

def start():
    global RECORDING
    RECORDING = True
    stop_button.config(state=tk.NORMAL)
    start_button.config(state=tk.DISABLED)
    record()

def stop():
    global RECORDING
    RECORDING = False
    stop_button.config(state=tk.DISABLED)
    start_button.config(state=tk.NORMAL)

def record():

    global prev_acc
    global prev_pressure
    global data_index
    global data
    global file_count
    global folder
    global data_size

    if RECORDING:
        if acc.getAcceleration() != prev_acc: # or barometer

            if data_size <= data_index:
                data = np.memmap(f"{folder}/{file_count}.dat", dtype=dtype, mode='w+', shape=(data_size,))
                file_count += 1
                data_index = 0

            data[data_index] = (
                acc.getTimestamp(),
                magnitude(acc.getAcceleration()) * sign(acc.getAcceleration()[2])
            )
            data_index += 1
            prev_acc = acc.getAcceleration()
            
        root.after(ceil(1000 / 60), record)

In [6]:
# Styling with ttk
style = ttk.Style()
style.configure("TButton", font=("Arial", 12), padding=10)
style.configure("TLabel", font=("Arial", 14))
style.configure("TFrame", background="#f0f0f0")

# Main frame
frame = ttk.Frame(root, padding=20, style="TFrame")
frame.pack(expand=True, fill=tk.BOTH)

# Title Label
title_label = ttk.Label(
    frame, text="Accelerometer + Barometer", font=("Arial", 16, "bold")
)
title_label.pack(pady=10)

# Status Label
status_label = ttk.Label(frame, text="Ready to Start", foreground="blue")
status_label.pack(pady=10)

# Buttons
button_frame = ttk.Frame(frame, padding=10, style="TFrame")
button_frame.pack(pady=20)

start_button = ttk.Button(button_frame, text="Start", command=start)
start_button.grid(row=0, column=0, padx=10)

stop_button = ttk.Button(button_frame, text="Stop", command=stop, state=tk.DISABLED)
stop_button.grid(row=0, column=1, padx=10)

# Run the application
root.mainloop()


In [7]:
acc.stop()

In [ ]:
# import numpy as np
# import os

# # Path to the memory-mapped file
# folder_path = './datastreams/'
# file_name = '04773f6e-7f26-4185-bc1d-720dee88f95c'
# file_path = os.path.join(folder_path, file_name)

# dtype = [('timestamp', 'int32'), ('acceleration', 'float32')]

# # Open the memory-mapped file in read-only mode
# data = np.memmap(file_path, dtype=dtype, mode='r')

# # Access elements
# print(data[1]['acceleration'])